# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alizawwaris974/Assignment-1---Flyrank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks" and os.path.basename(os.path.dirname(os.getcwd())) == "work":
    os.chdir("../..")  # from work/notebooks/ up two levels to repo root

import pandas as pd, numpy as np
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check working dir above"
print("Starter data found.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [4]:
#ranking coz ranks the page wrt some factors)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
visible = df[df["impressions_90d"] >= 100].copy()

# Proxy target: how far a page's CTR sits below its OWN position tier's average
tier_mean = visible.groupby("position_tier")["ctr"].transform("mean")
visible["ctr_gap_vs_tier"] = tier_mean - visible["ctr"]

visible[["content_id", "position_tier", "ctr", "ctr_gap_vs_tier"]].sort_values(
    "ctr_gap_vs_tier", ascending=False
).head(10)

,content_id,position_tier,ctr,ctr_gap_vs_tier
8939,content_478f26883850,page_1,0.0,0.35476
29977,content_c87291853cab,page_1,0.0,0.35476
29983,content_6880eb215048,page_1,0.0,0.35476
8873,content_268b56dc0221,page_1,0.0,0.35476
8915,content_e0ae71489787,page_1,0.0,0.35476
33,content_d87a116e2c79,page_1,0.0,0.35476
2282,content_3d2f81868b16,page_1,0.0,0.35476
24588,content_dc0d776ba8e5,page_1,0.0,0.35476
11665,content_50c7fe0d308f,page_1,0.0,0.35476
11673,content_6b5196195b13,page_1,0.0,0.35476


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [5]:
# Example metric: mean squared error of predicting a page's CTR gap vs. its tier average
# (a defensible "how good" number once you actually model ctr_gap_vs_tier)
baseline_pred = 0  # naive baseline: "assume every page matches its tier average exactly"
mse_baseline = ((visible["ctr_gap_vs_tier"] - baseline_pred) ** 2).mean()
print(f"Baseline MSE (predicting 0 gap for everyone): {mse_baseline:.5f}")
print(f"Mean absolute gap: {visible['ctr_gap_vs_tier'].abs().mean():.4f}")

Baseline MSE (predicting 0 gap for everyone): 0.15256
Mean absolute gap: 0.2257


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [6]:
# Example metric: mean squared error of predicting a page's CTR gap vs. its tier average
# (a defensible "how good" number once you actually model ctr_gap_vs_tier)
baseline_pred = 0  # naive baseline: "assume every page matches its tier average exactly"
mse_baseline = ((visible["ctr_gap_vs_tier"] - baseline_pred) ** 2).mean()
print(f"Baseline MSE (predicting 0 gap for everyone): {mse_baseline:.5f}")
print(f"Mean absolute gap: {visible['ctr_gap_vs_tier'].abs().mean():.4f}")

Baseline MSE (predicting 0 gap for everyone): 0.15256
Mean absolute gap: 0.2257


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [7]:
# One row = one (pseudonymized) content page, restricted to pages with real visibility
unit_of_analysis = visible[[
    "content_id", "client_id", "position_tier", "impressions_90d", "ctr", "ctr_gap_vs_tier"
]]
print(f"{len(unit_of_analysis)} rows -> one row per page")
unit_of_analysis.head(5)

22006 rows -> one row per page


,content_id,client_id,position_tier,impressions_90d,ctr,ctr_gap_vs_tier
0,content_304f48230142,client_f369cb89fc,striking,3803,0.76,-0.504218
1,content_a1fb4e703a9e,client_4e07408562,page_3_5,15320,0.05,0.092359
2,content_9aa793d4d895,client_7f2253d7e2,page_3_5,12581,0.09,0.052359
3,content_331d6c4de07b,client_19581e27de,page_1,11751,0.49,-0.135240
4,content_d99b7a2d90ca,client_3fdba35f04,page_3_5,19140,0.13,0.012359


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [8]:
# Show the within-tier spread a flat rule would miss
spread = visible.groupby("position_tier")["ctr"].agg(["mean", "std", "min", "max"])
print(spread.round(4))

                 mean     std  min    max
position_tier                            
deep           0.0554  0.1699  0.0   2.08
page_1         0.3548  0.5023  0.0  11.76
page_3_5       0.1424  0.2290  0.0   3.38
striking       0.2558  0.3467  0.0   5.43
top_3          0.3341  0.4875  0.0   6.19


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.